# पाठ ०१ - AI एजेन्टहरूको परिचय

**AI एजेन्टहरू शुरुवातकर्ताहरूका लागि** को पाठक्रमको पहिलो पाठमा स्वागत छ!

एक **AI एजेन्ट** एउटा कार्यक्रम हो जुन ठूलो भाषा मोडेल (LLM) लाई यसको तर्कशक्ति इन्जिनको रूपमा प्रयोग गर्छ र वास्तविक दुनियाँमा *कार्यहरू* लिन सक्छ — API सम्र्पक गर्ने, डाटाबेस क्वेरी गर्ने, वा कोड चलाउने — प्रयोगकर्ताको तर्फबाट लक्ष प्राप्त गर्न।

यस नोटबुकमा तपाईंले आफ्नो पहिलो एजेन्ट निर्माण गर्नु हुनेछ: एक **यात्रा एजेन्ट** जसले अवकाश गन्तव्यहरू सिफारिस गर्छ। यस क्रममा तपाईंले सिक्नु हुनेछ:

1. **Microsoft Agent Framework** को प्रयोग गरेर Azure AI Foundry Agent सेवा संग जडान कसरी गर्ने।
2. एजेन्टलाई एक **साधन** दिने — एक साधारण Python फङ्सन जुन यसले कल गर्न सक्छ।
3. एजेन्ट चलाउने र यसको प्रतिक्रिया निरीक्षण गर्ने।
4. एजेन्टको प्रतिक्रियालाई टोकन-द्वारा-टोकन स्ट्रिम गर्ने।


## सेटअप

यो नोटबुक चलाउनुअघि, पक्का गर्नुहोस् कि तपाईंसँग छ:

1. **एक Azure AI Foundry परियोजना** जुनमा एक च्याट मोडेल तैनाथ छ (जस्तै `gpt-4o-mini`)।
2. **Azure CLI मा लगइन गरिएको छ** — तपाईँको टर्मिनलमा `az login` चलाउनुहोस्।
3. **आवश्यक वातावरण चरहरू सेट गरिएको छ:**
   - `AZURE_AI_PROJECT_ENDPOINT` — तपाईंको Azure AI Foundry परियोजना इन्डपोइन्ट।
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — तपाईंको तैनाथ गरिएको मोडेलको नाम।

तलको सेलले आवश्यक Python प्याकेजहरू स्थापना गर्दछ।


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## तपाईँको पहिलो एजेन्ट सिर्जना गर्दै

एजेन्टलाई दुई चीजहरू आवश्यक पर्छन्:

- **निर्देशनहरू** जसले यसलाई *को हो* र *कसरी व्यवहार गर्ने* बताउँछ (एक सिस्टम प्रॉम्प्ट)।
- **उपकरणहरू** — Python कार्यहरू जसमा `@tool` द्वारा सजाइएको हुन्छन्, जसलाई एजेन्टले जानकारी प्राप्त गर्न वा क्रियाकलाप गर्न कल गर्न सक्छ।

तल हामीले एउटा सरल उपकरण परिभाषित गरेका छौं जुन लोकप्रिय छुट्टी गन्तव्यहरूको सूची फर्काउँछ। प्रयोगकर्ताले यात्रा सिफारिसहरूको लागि सोध्दा एजेन्टले यो उपकरण प्रयोग गर्नेछ।


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## स्ट्रिमिङ प्रतिक्रियाहरू

अझ अन्तरक्रियात्मक अनुभवको लागि तपाईँले एजेन्टको प्रतिक्रिया **स्ट्रिम** गर्न सक्नुहुन्छ। पूर्ण उत्तरको लागि कुर्नुको साटो, एजेन्टले उत्पन्न भएसँगै पाठका खण्डहरू प्रदान गर्छ। यो विशेष गरी च्याट इन्टरफेसहरूमा उपयोगी हुन्छ जहाँ तपाईँले वास्तविक समयमा आउटपुट देखाउन चाहनुहुन्छ।


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## सारांश

यस पाठमा तपाईंले यस्तो सिक्नुभयो:

- **प्रोभाइडर सिर्जना गर्ने** जुन `AzureAIProjectAgentProvider` मार्फत Azure AI Foundry Agent Service सँग जडान हुन्छ।
- **साधन परिभाषित गर्ने** `@tool` डेकोरेटर प्रयोग गरेर ताकि एजेन्टले तपाईंका Python फङ्सनहरूलाई कल गर्न सकोस्।
- **एजेन्ट चलाउने** प्रयोगकर्ता सन्देशसँग र यसको प्रतिक्रिया छाप्ने।
- **प्रतिक्रियाहरू स्ट्रिम गर्ने** वास्तविक-समय आउटपुटका लागि।

अर्को पाठमा हामी एजेन्टिक फ्रेमवर्कहरूलाई गहिराइमा अन्वेषण गर्नेछौं र एजेन्टहरूलाई अझ शक्तिशाली उपकरण र बहु-चरण तार्किक क्षमता दिने तरिका सिक्नेछौं।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**अस्वीकरण**:
यो दस्तावेज़ AI अनुवाद सेवा [Co-op Translator](https://github.com/Azure/co-op-translator) प्रयोग गरेर अनुवाद गरिएको हो। हामी शुद्धताको प्रयास गर्छौं, तर कृपया जान्नुहोस् कि स्वचालित अनुवादमा त्रुटि वा गलतफहमी हुन सक्छ। मूल भाषा मा रहेको दस्तावेज़लाई अधिकारिक स्रोत मान्नुपर्छ। महत्वपूर्ण जानकारीका लागि, व्यावसायिक मानवीय अनुवाद सिफारिस गरिन्छ। यस अनुवादको प्रयोगबाट उत्पन्न हुने कुनै पनि गलतफहमी वा गलत व्याख्याको लागि हामी जिम्मेवार छैनौं।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
